# VideoVeritas — Kaggle T4 GPU (Final Fix)

### Pre-requisites
1. **Internet ON** in Notebook settings
2. **GPU → T4 x2** accelerator
3. Video dataset added via **Add Input**

### Why previous attempts failed
| Attempt | Problem |
|---|---|
| Original notebook | `flashinfer` has no sm_75 wheel → runtime crash; wrong backend name `TRITON_ATTN` |
| v2 (pin vLLM 0.4.3) | 0.4.3 has no Python 3.12 wheel → pip builds from source → fails |
| v3 (no-deps install) | vLLM is a compiled extension; `--no-deps` doesn't help if the wheel itself doesn't exist |

### This notebook's approach (the correct one)
- Install **latest vLLM** — it ships pre-built Python 3.12 wheels ✓
- **Uninstall flashinfer** AFTER vLLM (flashinfer is optional; removing it is safe)
- Set `VLLM_ATTENTION_BACKEND=XFORMERS` — xformers supports T4 (sm_75) ✓
- `--enforce-eager` — disables CUDA graph capture (which needs flash-attn) ✓

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Install packages
# ═══════════════════════════════════════════════════════════════════════════════

# Step 1: Install vLLM (latest — has proper Python 3.12 wheels)
# This will also pull in flashinfer as a dependency, which is fine;
# we remove it in Step 2 before vLLM ever tries to USE it.
print('Installing vLLM and openai...')
!pip install -q vllm openai modelscope
print('✓ Install complete')

# Step 2: Remove flashinfer AFTER vLLM is installed.
# flashinfer has no sm_75 (T4) kernel, so it crashes vLLM at startup.
# vLLM treats it as optional — removing it forces vLLM to use xformers instead.
print('\nRemoving flashinfer (no T4/sm_75 support)...')
!pip uninstall -y flashinfer flashinfer-python 2>/dev/null; echo '✓ flashinfer removed'

# Step 3: Install xformers — the T4-compatible attention backend.
# Must be installed AFTER torch so it links against the right CUDA version.
print('\nInstalling xformers (T4-compatible attention backend)...')
!pip install -q xformers
print('✓ xformers installed')

# Step 4: Clone VideoVeritas repo
import os
if not os.path.isdir('/kaggle/working/VideoVeritas'):
    print('\nCloning VideoVeritas repo...')
    !git clone -q https://github.com/EricTan7/VideoVeritas.git /kaggle/working/VideoVeritas
    !cd /kaggle/working/VideoVeritas && pip install -q -e . --no-deps
else:
    print('\nVideoVeritas already cloned.')

# Verify
print('\n── Verification ──')
!python -c "import vllm; print('vLLM:', vllm.__version__)"
!python -c "import xformers; print('xformers:', xformers.__version__)"
!python -c "import torch; print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)"

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Download model
# ═══════════════════════════════════════════════════════════════════════════════
from modelscope import snapshot_download
import os

# The model is ~17 GB. ModelScope caches it under /root/.cache.
# We use that path DIRECTLY with vLLM — never copy it.
# /kaggle/working only has ~20 GB free; copying would cause disk-full errors.
CACHE_PATH = '/root/.cache/modelscope/hub/models/EricTanh/VideoVeritas'

if os.path.isdir(CACHE_PATH) and len(os.listdir(CACHE_PATH)) > 10:
    print(f'✓ Model already cached at:\n  {CACHE_PATH}')
    model_dir = CACHE_PATH
else:
    print('Downloading VideoVeritas model (~17 GB)…')
    model_dir = snapshot_download('EricTanh/VideoVeritas')
    print(f'✓ Model downloaded to: {model_dir}')

os.environ['MODEL_DIR'] = model_dir
print(f'\nMODEL_DIR = {model_dir}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — GPU info + libcuda stub
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess, os

# Detect available GPUs
res = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True
)
gpus = [l.strip() for l in res.stdout.strip().split('\n') if l.strip()]
NUM_GPUS = len(gpus)

print(f'Detected {NUM_GPUS} GPU(s):')
for g in gpus:
    print(f'  • {g}')

# Kaggle ships libcuda.so.1 but vLLM's linker looks for libcuda.so (no .1).
# Create a symlink so linking doesn't fail.
stub_dir = '/kaggle/working/stubs'
os.makedirs(stub_dir, exist_ok=True)
res2 = subprocess.run(
    ['find', '/usr', '/usr/local', '-name', 'libcuda.so.1', '-maxdepth', '12'],
    capture_output=True, text=True
)
paths = [p for p in res2.stdout.strip().split('\n') if p]
if paths:
    dst = f'{stub_dir}/libcuda.so'
    if not os.path.lexists(dst):
        os.symlink(paths[0], dst)
    print(f'✓ libcuda.so stub → {paths[0]}')
else:
    print('WARNING: libcuda.so.1 not found — proceeding anyway')

os.environ['NUM_GPUS']        = str(NUM_GPUS)
os.environ['LIBRARY_PATH']    = f'{stub_dir}:' + os.environ.get('LIBRARY_PATH', '')
os.environ['LD_LIBRARY_PATH'] = f'{stub_dir}:' + os.environ.get('LD_LIBRARY_PATH', '')
print(f'✓ tensor-parallel-size will be {NUM_GPUS}')

# Quick disk check
print('\n── Disk usage ──')
!df -h /root /kaggle/working

In [ ]:
import subprocess, sys, os

res = subprocess.run([sys.executable, '-c', 'import vllm; print(vllm.__version__)'],
                     capture_output=True, text=True)
vllm_version = res.stdout.strip()
print(f'vLLM version: {vllm_version}')

model_dir = os.environ['MODEL_DIR']
num_gpus  = os.environ['NUM_GPUS']
stub_dir  = '/kaggle/working/stubs'

res2 = subprocess.run(['which', 'vllm'], capture_output=True, text=True)
vllm_bin = res2.stdout.strip() or 'vllm'
print(f'vllm binary: {vllm_bin}')

script = f"""#!/usr/bin/env bash
set -euo pipefail

export VLLM_USE_V1=0
export VLLM_ATTENTION_BACKEND=XFORMERS
export VLLM_USE_FLASHINFER_SAMPLER=0
export VLLM_USE_FLASHINFER=0
export LIBRARY_PATH={stub_dir}:${{LIBRARY_PATH:-}}
export LD_LIBRARY_PATH={stub_dir}:${{LD_LIBRARY_PATH:-}}

echo "[vLLM] model   = {model_dir}"
echo "[vLLM] n_gpus  = {num_gpus}"
echo "[vLLM] backend = XFORMERS (T4 compatible)"

{vllm_bin} serve "{model_dir}" \\
    --enforce-eager \\
    --port 8000 \\
    --host 0.0.0.0 \\
    --dtype half \\
    --tensor-parallel-size {num_gpus} \\
    --max-model-len 32768 \\
    --gpu-memory-utilization 0.90 \\
    --limit-mm-per-prompt '{{"video": 1}}' \\
    --trust-remote-code
"""

path = '/kaggle/working/start_vllm.sh'
with open(path, 'w') as f:
    f.write(script)
os.chmod(path, 0o755)
print(f'\n✓ Launch script written')
print(script)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Launch vLLM server
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess, os, time

log_path = '/kaggle/working/vllm.log'

# Kill any existing instance
subprocess.run(['pkill', '-f', 'vllm'], capture_output=True)
time.sleep(2)

log_fh = open(log_path, 'w')
proc = subprocess.Popen(
    ['/kaggle/working/start_vllm.sh'],
    stdout=log_fh,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()   # pass ALL env vars set in previous cells
)

# Wait 5s — if it crashed instantly, show the log immediately
time.sleep(5)
log_fh.flush()

if proc.poll() is not None:
    print(f'✗ Server exited immediately (rc={proc.poll()}). Log:')
    with open(log_path) as f:
        print(f.read())
else:
    print(f'✓ vLLM server is starting (PID={proc.pid})')
    print(f'  Log file: {log_path}')
    print('\nFirst log lines:')
    with open(log_path) as f:
        print(f.read())
    print('\n→ Run Cell 6 to wait for the server to become ready (~3-5 min).')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Wait for server to be ready
# ═══════════════════════════════════════════════════════════════════════════════
import time, requests, subprocess

log_path = '/kaggle/working/vllm.log'
MAX_WAIT, INTERVAL = 600, 10

print(f'Polling http://localhost:8000/v1/models every {INTERVAL}s (up to {MAX_WAIT//60} min)...')
ready = False
for elapsed in range(0, MAX_WAIT, INTERVAL):
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=5)
        if r.status_code == 200:
            print(f'\n✓ Server READY after {elapsed}s!')
            print(r.json())
            ready = True
            break
    except Exception:
        pass
    time.sleep(INTERVAL)
    if elapsed % 60 == 0:
        print(f'\n── {elapsed}s elapsed — log tail ──')
        res = subprocess.run(['tail', '-n', '20', log_path], capture_output=True, text=True)
        print(res.stdout)

if not ready:
    print('\n✗ Timeout. Full log:')
    res = subprocess.run(['cat', log_path], capture_output=True, text=True)
    print(res.stdout[-10000:])

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6.5 — Expose the vLLM server to the internet with ngrok
# ═══════════════════════════════════════════════════════════════════════════════
# The vLLM server above only listens on localhost:8000, which the Verilens Chrome
# extension (running on your own machine) can't reach. ngrok opens a public HTTPS
# tunnel to it. Copy the printed URL into:
#     lib/config.local.js  ->  videoBackendUrl
#
# IMPORTANT if you also run fsd-image-detector.ipynb and/or fast-gpt.ipynb at the
# same time: ngrok's free plan assigns each ACCOUNT one shared static domain. If
# multiple notebooks use the same authtoken, the later tunnels fail with
# "ERR_NGROK_334: endpoint ... is already online". Fix: use a SEPARATE free
# ngrok account (and authtoken) for each notebook.
#   - This notebook reads the Kaggle Secret NGROK_AUTH_TOKEN_VIDEO (falls back
#     to NGROK_AUTH_TOKEN if that's the only one you've set up).
#   - fsd-image-detector.ipynb reads NGROK_AUTH_TOKEN_IMAGE.
#   - fast-gpt.ipynb reads NGROK_AUTH_TOKEN_TEXT.
#
# Get a free authtoken at: https://dashboard.ngrok.com/get-started/your-authtoken
# (Kaggle: add it as a Secret, or paste it inline below.)

!pip install -q pyngrok

import os
from pyngrok import ngrok

def _try_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ""

# Prefer a Kaggle Secret; fall back to the inline value.
NGROK_AUTH_TOKEN = ""  # ← paste your token here if not using Kaggle Secrets
NGROK_AUTH_TOKEN = (
    NGROK_AUTH_TOKEN
    or _try_secret("NGROK_AUTH_TOKEN_VIDEO")
    or _try_secret("NGROK_AUTH_TOKEN")
)

if not NGROK_AUTH_TOKEN:
    raise SystemExit("Set NGROK_AUTH_TOKEN_VIDEO (inline above or as a Kaggle Secret) and re-run this cell.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()  # drop any tunnel from a previous run
public_url = ngrok.connect(8000, "http").public_url

print("=" * 70)
print("VideoVeritas is now public at:")
print("   ", public_url)
print("=" * 70)
print("\nPaste that URL into the Verilens extension:")
print("   lib/config.local.js -> videoBackendUrl")
print("\nSanity check (should list the model):")
print(f"   curl {public_url}/v1/models -H 'ngrok-skip-browser-warning: true'")


## Inference
Update `video_path` to a file from your Kaggle Input dataset.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SKIPPED — optional single-video sanity test, not needed to serve the API (the
# ngrok URL printed by the cell above is already live). Re-enable only if you
# want to manually verify inference works before connecting the extension.
# ═══════════════════════════════════════════════════════════════════════════════

# import base64, os, re
# from openai import OpenAI

# model_dir = os.environ.get('MODEL_DIR', '/root/.cache/modelscope/hub/models/EricTanh/VideoVeritas')
# client    = OpenAI(api_key='EMPTY', base_url='http://localhost:8000/v1')

# SYSTEM = (
#     'You are an expert video analyst. Think carefully. '
#     'Give your final verdict inside <answer> </answer> tags.'
# )

# # ↓↓↓ CHANGE THIS PATH ↓↓↓
# video_path = '/kaggle/input/datasets/kanzeus/realai-video-dataset/real/real (23).mp4'

# if not os.path.exists(video_path):
#     print(f"Not found: '{video_path}'")
#     print("Available videos:")
#     !find /kaggle/input -name '*.mp4' -o -name '*.avi' | head -20
# else:
#     ext = os.path.splitext(video_path)[1].lstrip('.') or 'mp4'
#     print(f'Reading {os.path.getsize(video_path)/1e6:.1f} MB: {video_path}')
#     with open(video_path, 'rb') as f:
#         b64 = base64.b64encode(f.read()).decode()

#     resp = client.chat.completions.create(
#         model=model_dir,
#         messages=[
#             {'role': 'system', 'content': SYSTEM},
#             {'role': 'user', 'content': [
#                 {'type': 'video_url', 'video_url': {'url': f'data:video/{ext};base64,{b64}'}},
#                 {'type': 'text', 'text': 'Is this video real or AI-generated/fake?'},
#             ]},
#         ],
#         temperature=0.7,
#         extra_body={
#             'mm_processor_kwargs': {
#                 'fps': 2,
#                 # max_pixels caps each frame's resolution before tokenisation.
#                 # 360x420 = 151200 px/frame keeps token count manageable on T4.
#                 # Qwen3VL accepts: fps, min_pixels, max_pixels, total_pixels
#                 # do_sample_frames is NOT valid for Qwen3VL -- removing it was the fix.
#                 'max_pixels': 360 * 420,
#             }
#         },
#     )
#     content = resp.choices[0].message.content
#     print('\n--- RESPONSE ---\n', content)
#     m = re.search(r'<answer>(.*?)</answer>', content, re.DOTALL)
#     if m:
#         print(f'\nVERDICT: {m.group(1).strip()}')


In [ ]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # CELL 8 — Batch inference
# # ═══════════════════════════════════════════════════════════════════════════════
# import base64, os, json, re
# from pathlib import Path
# from openai import OpenAI

# model_dir = os.environ.get('MODEL_DIR', '/root/.cache/modelscope/hub/models/EricTanh/VideoVeritas')
# client    = OpenAI(api_key='EMPTY', base_url='http://localhost:8000/v1')

# # ↓↓↓ Change this path ↓↓↓
# VIDEO_FOLDER = '/kaggle/input/your-dataset/'
# EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

# videos = [p for p in Path(VIDEO_FOLDER).rglob('*') if p.suffix.lower() in EXTS]
# if not videos:
#     print(f'No videos found in {VIDEO_FOLDER}')
# else:
#     results = []
#     for vp in videos:
#         print(f'{vp.name}... ', end='', flush=True)
#         try:
#             with open(vp, 'rb') as f:
#                 b64 = base64.b64encode(f.read()).decode()
#             ext = vp.suffix.lstrip('.')
#             resp = client.chat.completions.create(
#                 model=model_dir,
#                 messages=[
#                     {'role': 'system', 'content': 'Analyze video. Verdict in <answer></answer>.'},
#                     {'role': 'user', 'content': [
#                         {'type': 'video_url', 'video_url': {'url': f'data:video/{ext};base64,{b64}'}},
#                         {'type': 'text', 'text': 'Real or AI-generated?'},
#                     ]},
#                 ],
#                 temperature=0.3,
#                 extra_body={'mm_processor_kwargs': {'fps': 2, 'do_sample_frames': True}},
#             )
#             content = resp.choices[0].message.content
#             m = re.search(r'<answer>(.*?)</answer>', content, re.DOTALL)
#             verdict = m.group(1).strip() if m else 'UNKNOWN'
#             results.append({'file': str(vp), 'verdict': verdict})
#             print(verdict)
#         except Exception as e:
#             print(f'ERROR: {e}')
#             results.append({'file': str(vp), 'verdict': 'ERROR', 'error': str(e)})

#     out = '/kaggle/working/results.json'
#     with open(out, 'w') as f:
#         json.dump(results, f, indent=2)
#     print(f'\n✓ Saved to {out}')
#     import pandas as pd
#     display(pd.DataFrame(results))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9 — Diagnostics (run any time)
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess

print('='*60)
print('VLLM LOG')
print('='*60)
!cat /kaggle/working/vllm.log

print('\n' + '='*60)
print('ENVIRONMENT')
print('='*60)
!python -c "import vllm; print('vLLM:', vllm.__version__)"
!python -c "import torch; print('torch:', torch.__version__, 'CUDA:', torch.version.cuda)"
!python -c "import xformers; print('xformers:', xformers.__version__)"
!python -m pip show flashinfer 2>/dev/null && echo 'WARNING: flashinfer still installed!' || echo '✓ flashinfer not present'

print('\n' + '='*60)
print('GPU')
print('='*60)
!nvidia-smi

print('\n' + '='*60)
print('DISK')
print('='*60)
!df -h /root /kaggle/working